# 02 — Chunk & Embed
## Databricks Expert Agent Project

**What this notebook does:**
1. Reads raw docs from `chatbot.rag_chatbot.raw_docs`
2. Splits each page into overlapping chunks (~1,000 chars, 200 overlap)
3. Embeds each chunk using Databricks Foundation Model API (gte-large-en)
4. Writes chunks + embeddings to `chatbot.rag_chatbot.doc_chunks`

**Output:** A Delta table with one row per chunk, including a 1,024-dimension embedding vector — ready for Vector Search indexing in Notebook 3.

## 1. Configure Chunking And Embedding Targets

This cell defines the raw input table, the chunk output table, and the embedding model used by the Databricks corpus pipeline.

The chunking settings are part of the retrieval quality contract: chunks need to be small enough to retrieve precisely, but large enough to preserve useful surrounding context.

In [0]:
# ── Config ───────────────────────────────────────────────────────
CATALOG      = "chatbot"
SCHEMA       = "rag_chatbot"
RAW_TABLE    = f"{CATALOG}.{SCHEMA}.raw_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.doc_chunks"

CHUNK_SIZE    = 1000   # characters per chunk
CHUNK_OVERLAP = 200    # overlap between consecutive chunks
EMBED_MODEL   = "databricks-gte-large-en"   # Foundation Model API endpoint
EMBED_BATCH   = 25     # max inputs per API call (model limit)

print(f"Source  : {RAW_TABLE}")
print(f"Target  : {CHUNKS_TABLE}")
print(f"Chunks  : {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")
print(f"Model   : {EMBED_MODEL}")

## 2. Define The Chunking Function

This function splits long documentation pages into overlapping text windows. The overlap is intentional: it helps preserve context when an important sentence falls near a chunk boundary.

Rows with extremely short or empty content are skipped because they are usually navigation pages, scrape artifacts, or weak retrieval candidates.

In [0]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list:
    # Skip blank or tiny pages. Very short text usually adds noise to retrieval.
    if not text or len(text) < 100:
        return []

    chunks = []
    step = chunk_size - overlap
    start = 0

    # Slide a fixed-size window through the page. The overlap keeps context
    # around chunk boundaries, which improves retrieval quality for long docs.
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if len(chunk.strip()) > 100:
            chunks.append(chunk.strip())
        start += step

    return chunks

## 3. Build Chunk Rows With Source Metadata

This cell turns each raw document into multiple chunk rows and keeps metadata that the app can use later.

Key outputs from this cell:

- `chunk_text`: the retrievable text segment.
- `chunk_index`: the chunk position within the original page.
- `source` and `source_type`: useful for separating official docs from internal wiki content.
- `chunk_id`: a stable primary key for Vector Search.

In [0]:
from pyspark.sql.functions import udf, posexplode, concat, lit, col, when
from pyspark.sql.types import ArrayType, StringType

# Wrap chunker as a UDF that returns an array of strings
chunk_udf = udf(lambda text: chunk_text(text) if text else [], ArrayType(StringType()))

# Read raw docs
raw_df = spark.table(RAW_TABLE)

# Add source metadata if old raw_docs table does not have it
if "source" not in raw_df.columns:
    raw_df = raw_df.withColumn(
        "source",
        when(col("url").startswith("internal_wiki://"), lit("internal_wiki"))
        .otherwise(lit("microsoft_learn_azure_databricks"))
    )

if "cloud" not in raw_df.columns:
    raw_df = raw_df.withColumn("cloud", lit("azure"))

# Apply chunker — produces an array of chunk strings per row
# posexplode turns array into multiple rows and gives us position (pos) of each chunk
chunks_df = (
    raw_df
    .select(
        "url",
        "title",
        "scraped_date",
        "source",
        "cloud",
        chunk_udf("content").alias("chunks")
    )
    .select(
        "url",
        "title",
        "scraped_date",
        "source",
        "cloud",
        posexplode("chunks").alias("chunk_index", "chunk_text")
    )
    .filter("length(chunk_text) > 100")
)

# Add a source_type to help the app rank internal runbooks higher later
chunks_df = chunks_df.withColumn(
    "source_type",
    when(col("source") == "internal_wiki", lit("internal"))
    .otherwise(lit("official_docs"))
)

# Add a unique chunk_id — url + position so it's human readable and unique
chunks_df = chunks_df.withColumn(
    "chunk_id",
    concat(col("url"), lit("::chunk::"), col("chunk_index").cast("string"))
)

print(f"Total chunks : {chunks_df.count():,}")
print(f"Avg per page : {chunks_df.count() / raw_df.count():.1f}")

display(chunks_df.select(
    "chunk_id",
    "url",
    "title",
    "source",
    "source_type",
    "cloud",
    "chunk_index",
    "chunk_text"
).limit(5))

## 4. Smoke Test The Embedding Endpoint

Before embedding the full corpus, this cell sends one short test prompt to the embedding endpoint.

This catches endpoint, authentication, permission, or model-name problems early. It is much cheaper to fail here than after launching a full distributed embedding job.

In [0]:
import mlflow.deployments

deploy_client = mlflow.deployments.get_deploy_client("databricks")

try:
    test_response = deploy_client.predict(
        endpoint=EMBED_MODEL,
        inputs={"input": ["What is Delta Lake and how does it work?"]}
    )
    embedding = test_response.data[0]["embedding"]
    print(f"✓ API call succeeded")
    print(f"✓ Embedding dimensions : {len(embedding)}")
    print(f"✓ First 5 values       : {embedding[:5]}")
except Exception as e:
    raise RuntimeError(f"Embedding endpoint test failed for '{EMBED_MODEL}': {e}")

## 5. Define A Distributed Embedding UDF

This cell defines a pandas UDF that embeds many chunk rows from Spark.

The UDF batches calls to the Databricks model-serving endpoint in groups of 25, which matches the expected endpoint input shape and avoids sending one HTTP request per row. Failed batches return `None` embeddings so the next cell can filter them out and report the gap.

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import ArrayType, FloatType
import mlflow.deployments
import time

EMBED_MODEL  = "databricks-gte-large-en"
EMBED_BATCH  = 25

@pandas_udf(ArrayType(FloatType()))
def embed_udf(texts: pd.Series) -> pd.Series:
    """
    Pandas UDF — receives a batch of text strings, 
    calls the embedding API in sub-batches of 25,
    returns a list of 1024-float vectors.
    """
    client = mlflow.deployments.get_deploy_client("databricks")
    results = []
    batch = texts.tolist()

    # Split into sub-batches of EMBED_BATCH size
    for i in range(0, len(batch), EMBED_BATCH):
        sub_batch = batch[i : i + EMBED_BATCH]
        try:
            resp = client.predict(
                endpoint=EMBED_MODEL,
                inputs={"input": sub_batch}
            )
            for item in resp.data:
                results.append(item["embedding"])
        except Exception as e:
            # On failure, append None vectors so we don't lose the row
            for _ in sub_batch:
                results.append(None)
            print(f"Embedding batch {i} failed: {e}")
        time.sleep(0.1)   # brief pause to avoid rate limit spikes

    return pd.Series(results)

print("✓ Embed UDF defined — ready to run")

## 6. Embed Chunks And Write The Delta Table

This cell applies the embedding UDF to every chunk, removes failed embedding rows, and overwrites the Databricks chunk table.

The output table is the source table for the Vector Search Delta Sync index in Notebook 3. Treat this as a publication step: once this table changes, the retrieval index needs to be refreshed or synced.

In [0]:
print(f"Embedding {chunks_df.count():,} chunks — expect 15-25 minutes...\n")

# Apply the embedding UDF to add a vector column
embedded_df = chunks_df.withColumn("embedding", embed_udf(col("chunk_text")))

# Drop any rows where embedding failed
embedded_df = embedded_df.filter(col("embedding").isNotNull())

# Write to Delta
(
    embedded_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CHUNKS_TABLE)
)

final_count = spark.table(CHUNKS_TABLE).count()
print(f"✓ Written to : {CHUNKS_TABLE}")
print(f"✓ Row count  : {final_count:,}")

## 7. Validate Embedding Coverage

This cell checks the resulting chunk table after the write.

The key quality checks are:

- Chunk count is nonzero.
- No embeddings are null.
- The first embedding has the expected vector length.
- Sample chunks look like useful documentation text rather than navigation noise.

In [0]:
from pyspark.sql.functions import col

df = spark.table(CHUNKS_TABLE)
df.printSchema()

print(f"\nTotal chunks     : {df.count():,}")
print(f"Null embeddings  : {df.filter(col('embedding').isNull()).count()}")
print(f"Embedding length : {len(df.select('embedding').first()[0])}")

display(df.select("title", "chunk_index", "chunk_text").limit(3))

## 8. Reserved Scratch Cell

This final empty cell is intentionally left available for temporary ad hoc checks while developing the pipeline. Keep production logic in the named cells above so the notebook remains easy to review.